# 3 — Sanity: scANVI predictions vs per-dataset CellAssign

For each query that already has a CellAssign result (D2_DZ, D2_Lapa, D4_Lapa — D4_DZ has no CellAssign notebook), build a confusion matrix and report agreement.

Also plots a marker-gene dotplot per query keyed on `scanvi_prediction`: predicted Goblet should express MUC2/FCGBP/GFI1, predicted EECs CHGA/CHGB, predicted Enterocytes KRT20/FABP2/ALPI.

In [ ]:
import sys, gc
from pathlib import Path

_p = Path('.').resolve()
while not (_p / 'src' / 'config.py').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

from src.config import (
    DATASETS, CELLASSIGN_DIR, FIGURES_DIR, UTILITIES_DIR,
)
from src.cell_assign import compute_size_factors, run_cellassign, annotate_predictions
from src.markers import cell_type_markers

import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

QUERIES_WITH_CELLASSIGN = ['D2_DZ', 'D2_Lapa', 'D4_Lapa']
MARKER_CSV = UTILITIES_DIR / 'cellassign_markers.csv'

fig_dir = FIGURES_DIR / 'd2-d4-integrated' / 'sanity'
fig_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_per_dataset_cellassign(key: str) -> pd.Series:
    """Return per-dataset CellAssign labels for a query, indexed by cell barcode.

    Tries the labelled h5ad first (cheaper), falls back to running CellAssign.
    """
    labelled_path = Path(DATASETS[key]['labelled'])
    if labelled_path.exists():
        a = sc.read_h5ad(labelled_path)
        for col in ('initial_cellassign_prediction', 'manual_label', 'cellassign_label'):
            if col in a.obs.columns:
                print(f'{key}: using {col} from {labelled_path.name}')
                out = a.obs[col].astype(str).copy()
                del a
                return out
        del a
    print(f'{key}: re-running CellAssign')
    a = sc.read_h5ad(DATASETS[key]['clustered'])
    a = compute_size_factors(a, layer='counts')
    preds = run_cellassign(a, MARKER_CSV, layer='counts', seed=0)
    a = annotate_predictions(a, preds)
    out = a.obs['initial_cellassign_prediction'].astype(str).copy()
    del a, preds
    gc.collect()
    return out

In [ ]:
agreement_summary = []
for key in QUERIES_WITH_CELLASSIGN:
    print(f'\n=== {key} ===')
    scanvi_h5ad = CELLASSIGN_DIR / f'{key.lower()}_scanvi_predictions.h5ad'
    a = sc.read_h5ad(scanvi_h5ad)
    scanvi_pred = a.obs['scanvi_prediction_raw'].astype(str)  # use raw (no Ambiguous_ prefix) for crosstab

    cellassign_pred = get_per_dataset_cellassign(key)
    common = scanvi_pred.index.intersection(cellassign_pred.index)
    sp, cp = scanvi_pred.loc[common], cellassign_pred.loc[common]

    ct = pd.crosstab(cp, sp, normalize='index').round(3)
    print(ct)

    agree = (sp == cp).mean()
    print(f'overall agreement: {agree:.3%}')

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(ct, cmap='viridis', annot=True, fmt='.2f', ax=ax)
    ax.set_title(f'{key}: row-normalized CellAssign vs scANVI argmax (agreement={agree:.1%})')
    ax.set_xlabel('scANVI raw argmax'); ax.set_ylabel('per-dataset CellAssign')
    plt.tight_layout()
    plt.savefig(fig_dir / f'{key.lower()}_crosstab.pdf')
    plt.show()

    agreement_summary.append({'dataset': key, 'agreement': agree, 'n': len(common)})
    del a; gc.collect()

pd.DataFrame(agreement_summary)

In [ ]:
# Marker dotplots: each query's predicted classes should still show canonical markers.
# We dotplot on the per-query scANVI h5ad (counts on HVGs only — most epithelial markers should be in the HVG set).
for key in ['D2_DZ', 'D2_Lapa', 'D4_DZ', 'D4_Lapa']:
    a = sc.read_h5ad(CELLASSIGN_DIR / f'{key.lower()}_scanvi_predictions.h5ad')
    # ensure log-normalized .X for plotting (keep counts in layer)
    if 'counts' in a.layers and (a.X is None or a.X.shape != a.layers['counts'].shape):
        a.X = a.layers['counts']
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)

    present = {ct: [g for g in genes if g in a.var_names] for ct, genes in cell_type_markers.items()}
    present = {ct: g for ct, g in present.items() if g}

    fig, ax = plt.subplots(figsize=(10, 4))
    sc.pl.dotplot(a, present, groupby='scanvi_prediction_raw', ax=ax, show=False)
    plt.suptitle(f'{key}: marker expression by scANVI argmax')
    plt.tight_layout()
    plt.savefig(fig_dir / f'{key.lower()}_marker_dotplot.pdf')
    plt.show()
    del a; gc.collect()